In [10]:
# ========================================================
# BAGIAN 0: Install library (jika belum ada)
# ========================================================
!pip install -q imagehash Pillow opencv-python matplotlib

import os
import hashlib
import imagehash
from PIL import Image
from collections import defaultdict
import re
from pathlib import Path
import datetime


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# ========================================================
# KONFIGURASI PATH
# ========================================================
PROJECT_ROOT = Path.cwd().parent   # karena notebook ada di 'notebooks/'
DATA_DIR = PROJECT_ROOT / "data"
DATASET_DIR = DATA_DIR / "dataset"      # folder asli, palsu
DEMO_DIR = DATA_DIR / "demo_test"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATASET_DIR  : {DATASET_DIR}")
print(f"DEMO_DIR     : {DEMO_DIR}")

PROJECT_ROOT : e:\AIC\BlockchainAI
DATASET_DIR  : e:\AIC\BlockchainAI\data\dataset
DEMO_DIR     : e:\AIC\BlockchainAI\data\demo_test


In [12]:
# ========================================================
# FUNGSI: DETEKSI DAN HAPUS DUPLIKAT (EXACT & NEAR)
# ========================================================
def find_duplicates_in_folder(folder_path, similarity_threshold=5):
    """
    Mencari gambar duplikat atau sangat mirip di dalam folder.
    Return: (list_of_duplicate_groups, list_of_files_to_remove)
    """
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return [], []

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    file_paths = [os.path.join(folder_path, f) for f in files]

    # Step 1: Exact duplicate (MD5)
    hash_dict = defaultdict(list)
    for fp in file_paths:
        with open(fp, 'rb') as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        hash_dict[md5].append(fp)

    exact_duplicates = [group for group in hash_dict.values() if len(group) > 1]

    files_to_remove = set()
    for group in exact_duplicates:
        group_sorted = sorted(group)
        for dup in group_sorted[1:]:
            files_to_remove.add(dup)

    # Step 2: Near-duplicate (perceptual hash)
    remaining_files = [fp for fp in file_paths if fp not in files_to_remove]
    hashes = {}
    for fp in remaining_files:
        try:
            img = Image.open(fp)
            hashes[fp] = imagehash.phash(img)
        except Exception:
            continue

    near_duplicate_groups = []
    processed = set()
    fps = list(hashes.keys())
    for i in range(len(fps)):
        if fps[i] in processed:
            continue
        group = [fps[i]]
        for j in range(i+1, len(fps)):
            if fps[j] in processed:
                continue
            dist = hashes[fps[i]] - hashes[fps[j]]
            if dist <= similarity_threshold:
                group.append(fps[j])
                processed.add(fps[j])
        if len(group) > 1:
            near_duplicate_groups.append(group)
            processed.add(fps[i])

    for group in near_duplicate_groups:
        group_sorted = sorted(group)
        for dup in group_sorted[1:]:
            files_to_remove.add(dup)

    all_duplicate_groups = exact_duplicates + near_duplicate_groups
    return all_duplicate_groups, list(files_to_remove)

In [13]:
# ========================================================
# PROSES: DUPLIKAT DI FOLDER ASLI
# ========================================================
print("="*60)
print("MEMERIKSA DUPLIKAT DI FOLDER ASLI")
print("="*60)

asli_dir = DATASET_DIR / "asli"
removed_asli = []
if not asli_dir.exists():
    print(f"Folder tidak ditemukan: {asli_dir}")
else:
    dup_groups, to_remove = find_duplicates_in_folder(str(asli_dir))
    if dup_groups:
        print(f"Ditemukan {len(dup_groups)} grup duplikat.")
        for i, group in enumerate(dup_groups):
            print(f"Grup {i+1}: {len(group)} file")
            for fp in group:
                print(f"  - {fp}")
    else:
        print("Tidak ada duplikat ditemukan.")

    if to_remove:
        print(f"\nMenghapus {len(to_remove)} file duplikat...")
        for fp in to_remove:
            os.remove(fp)
            removed_asli.append(fp)
            print(f"Dihapus: {fp}")
    else:
        print("Tidak ada file yang dihapus.")

MEMERIKSA DUPLIKAT DI FOLDER ASLI
Ditemukan 441 grup duplikat.
Grup 1: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\00db58cf-53b0-4840-a170-466a061a5594.jpg
  - e:\AIC\BlockchainAI\data\dataset\asli\158b7b05-5feb-48b6-b737-6170d77d7294.jpg
Grup 2: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\02771f56-7998-494f-8b73-10533e217159.jpg
  - e:\AIC\BlockchainAI\data\dataset\asli\22a31f29-279b-49dc-9335-6b23837cc764.jpg
Grup 3: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\031d3571-fc54-4b86-b4d1-713ddaac4270.jpg
  - e:\AIC\BlockchainAI\data\dataset\asli\idugf.jpeg
Grup 4: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\05cfaec1-4f34-4c88-b3fe-6069c567a66e.jpg
  - e:\AIC\BlockchainAI\data\dataset\asli\a8207d0e-5264-4ea0-9fc8-ae36c5d81b59.jpg
Grup 5: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\05de4778-3a7c-4cc4-b9e6-a4f80810f576.jpg
  - e:\AIC\BlockchainAI\data\dataset\asli\294e92f5-7193-4f5e-a220-47274294f4a7.jpg
Grup 6: 2 file
  - e:\AIC\BlockchainAI\data\dataset\asli\07969

In [14]:
# ========================================================
# PROSES: DUPLIKAT DI FOLDER PALSU
# ========================================================
print("\n" + "="*60)
print("MEMERIKSA DUPLIKAT DI FOLDER PALSU")
print("="*60)

palsu_dir = DATASET_DIR / "palsu"
removed_palsu = []
if not palsu_dir.exists():
    print(f"Folder tidak ditemukan: {palsu_dir}")
else:
    dup_groups, to_remove = find_duplicates_in_folder(str(palsu_dir))
    if dup_groups:
        print(f"Ditemukan {len(dup_groups)} grup duplikat.")
        for i, group in enumerate(dup_groups):
            print(f"Grup {i+1}: {len(group)} file")
            for fp in group:
                print(f"  - {fp}")
    else:
        print("Tidak ada duplikat ditemukan.")

    if to_remove:
        print(f"\nMenghapus {len(to_remove)} file duplikat...")
        for fp in to_remove:
            os.remove(fp)
            removed_palsu.append(fp)
            print(f"Dihapus: {fp}")
    else:
        print("Tidak ada file yang dihapus.")


MEMERIKSA DUPLIKAT DI FOLDER PALSU
Ditemukan 155 grup duplikat.
Grup 1: 2 file
  - e:\AIC\BlockchainAI\data\dataset\palsu\0167615b-ffb2-44e5-be70-469f328de3e0.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\fhjbdslf_2_.jpeg
Grup 2: 3 file
  - e:\AIC\BlockchainAI\data\dataset\palsu\0b148581-ba6b-4109-9042-1d767924ed92.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\aab777e9-f6de-4243-8f99-db5a0dfd8ea9.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\palsu4(1).jpeg
Grup 3: 2 file
  - e:\AIC\BlockchainAI\data\dataset\palsu\0c24f66b-af2e-4711-80ac-7075a7796eb7.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\PALSU (5).jpeg
Grup 4: 3 file
  - e:\AIC\BlockchainAI\data\dataset\palsu\138e6820-0790-4ee1-9b51-d95048e5d0da.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\Copy_of_PALSU2.jpeg
  - e:\AIC\BlockchainAI\data\dataset\palsu\PALSU2.jpeg
Grup 5: 2 file
  - e:\AIC\BlockchainAI\data\dataset\palsu\19f18231-219a-458d-9c26-4f38d4bcf2f4.jpg
  - e:\AIC\BlockchainAI\data\dataset\palsu\fhjbdslf_1_.jpeg
Gr

In [15]:
# ========================================================
# FUNGSI: MEMBERSIHKAN NAMA FILE
# ========================================================
def clean_filenames(folder_path):
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return 0
    count = 0
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            nama_baru = re.sub(r'[^a-zA-Z0-9_.]', '_', fname)
            nama_baru = re.sub(r'_+', '_', nama_baru)
            if nama_baru != fname:
                src = os.path.join(folder_path, fname)
                dst = os.path.join(folder_path, nama_baru)
                os.rename(src, dst)
                count += 1
                print(f"Renamed: {fname} → {nama_baru}")
    print(f"Total file di-rename: {count}")
    return count

In [16]:
# ========================================================
# PROSES: PEMBERSIHAN NAMA FILE DI SEMUA FOLDER
# ========================================================
print("\n" + "="*60)
print("MEMBERSIHKAN NAMA FILE")
print("="*60)

renamed_total = 0
for folder_name, folder_path in [('asli', DATASET_DIR / "asli"),
                                 ('palsu', DATASET_DIR / "palsu"),
                                 ('demo_test', DEMO_DIR)]:
    if folder_path.exists():
        print(f"\nMembersihkan folder: {folder_name}")
        renamed_total += clean_filenames(str(folder_path))
    else:
        print(f"Folder {folder_name} tidak ditemukan, dilewati.")

print(f"\n✅ Total file di-rename: {renamed_total}")


MEMBERSIHKAN NAMA FILE

Membersihkan folder: asli
Renamed: 00db58cf-53b0-4840-a170-466a061a5594.jpg → 00db58cf_53b0_4840_a170_466a061a5594.jpg
Renamed: 01cdbe45-ad85-4ce6-8334-d33e86da7e2b.jpg → 01cdbe45_ad85_4ce6_8334_d33e86da7e2b.jpg
Renamed: 02413c3b-78f6-4716-b9f8-c7b29d73b043.jpg → 02413c3b_78f6_4716_b9f8_c7b29d73b043.jpg
Renamed: 02771f56-7998-494f-8b73-10533e217159.jpg → 02771f56_7998_494f_8b73_10533e217159.jpg
Renamed: 031d3571-fc54-4b86-b4d1-713ddaac4270.jpg → 031d3571_fc54_4b86_b4d1_713ddaac4270.jpg
Renamed: 05cfaec1-4f34-4c88-b3fe-6069c567a66e.jpg → 05cfaec1_4f34_4c88_b3fe_6069c567a66e.jpg
Renamed: 05de4778-3a7c-4cc4-b9e6-a4f80810f576.jpg → 05de4778_3a7c_4cc4_b9e6_a4f80810f576.jpg
Renamed: 0796914e-b6c4-48c9-bca5-ca463e44c2cd.jpg → 0796914e_b6c4_48c9_bca5_ca463e44c2cd.jpg
Renamed: 085a7f34-1872-443d-b551-aa11da8c8ebd.jpg → 085a7f34_1872_443d_b551_aa11da8c8ebd.jpg
Renamed: 08a7e979-df50-489f-8955-f590ff791907.jpg → 08a7e979_df50_489f_8955_f590ff791907.jpg
Renamed: 0aa2e133-e

In [17]:
# ========================================================
# SIMPAN LOG PEMBERSIHAN (OPSIONAL)
# ========================================================
log_path = OUTPUT_DIR / "clean_log.txt"
with open(log_path, 'w') as f:
    f.write(f"Log pembersihan - {datetime.datetime.now()}\n")
    f.write(f"Folder dataset: {DATASET_DIR}\n\n")
    f.write("--- Duplikat dihapus (asli) ---\n")
    for fp in removed_asli:
        f.write(f"{fp}\n")
    f.write("\n--- Duplikat dihapus (palsu) ---\n")
    for fp in removed_palsu:
        f.write(f"{fp}\n")
    f.write(f"\nTotal file di-rename: {renamed_total}\n")
print(f"Log disimpan ke {log_path}")

Log disimpan ke e:\AIC\BlockchainAI\output\clean_log.txt
